# Lab L4.1 · SVMs in production conditions — tuned RBF fraud classifier, kernel approximation at scale, and SVR for loss severity
**Horizon Gulf Bank · Project MIZAN** · CAIP AIP-210 · Certified AI Practitioner · Day 4 · ⏱ 70 min · **Level 1 · From scratch**

Tune C and γ properly, hit the wall of kernel-SVM complexity, break through it with a Nyström approximation, and apply SVR to predict loss amounts on defaulted loans.

Follow **L4.1_Lab_Guide.html** step by step. This notebook gives you only the structure — every line of logic is yours. Keep variable names as suggested in the guide so later steps (and later labs) line up.

In [ ]:
# ── Project MIZAN course setup ─ run this cell first ──────────────────────────
from pathlib import Path
import warnings, time
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Locate the course root (the folder that contains data/generate_data.py) from wherever this notebook lives
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "generate_data.py").exists())
DATA = ROOT / "data"          # raw synthetic datasets (read-only)
CKPT = ROOT / "checkpoints"   # hand-off artefacts between labs (lets you start any lab fresh)
CKPT.mkdir(exist_ok=True)
pd.set_option("display.max_columns", 60); pd.set_option("display.width", 160)
plt.rcParams.update({"figure.figsize": (9, 4.5), "axes.grid": True, "grid.alpha": .3})
RNG = 42
print("Course root:", ROOT)

## Step 1 · Load L3.3 features and define the evaluation

1. Load `card_features.parquet` and `fraud_metrics.json`; split into `tr/va/te`.
2. Re-create `at_budget(y, score, amount)` — it must work with any score (SVM decision values are not probabilities).

In [ ]:
# Your code for step 1


## Step 2 · Preprocessing for SVMs and a LinearSVC baseline

1. Build `svm_prep`: numeric → impute −1 → `StandardScaler`; categorical → `OneHotEncoder`.
2. Create `tr_u` = all training fraud + 30k legit.
3. Fit `LinearSVC(C=0.1, class_weight='balanced')`; time it; evaluate `decision_function` on validation with `at_budget`.

In [ ]:
# Your code for step 2


## Step 3 · Tune an RBF SVM with cross-validation on a stratified sample

1. Draw a 6k stratified sample of `tr_u`.
2. `GridSearchCV` over C ∈ {0.3, 1, 3, 10} and γ ∈ {0.01, 0.03, 0.1, 0.3}, `scoring='average_precision'`, `cv=3`, `n_jobs=-1`. Show the results as a C × γ heat-map table.
3. Refit `SVC(kernel='rbf')` with the best parameters on all of `tr_u`; time fit and validation scoring; add to `board`.

In [ ]:
# Your code for step 3


## Step 4 · Kernel approximation: RBF-like power at linear cost

1. Build `Pipeline(svm_prep, Nystroem(kernel='rbf', gamma=best γ, n_components=600), LinearSVC(C=0.5, class_weight='balanced'))`.
2. Train on all training fraud + 250k legit (`tr_big`); time it; evaluate.
3. Add a final row with the RF benchmark from `fm` and compare.

In [ ]:
# Your code for step 4


## Step 5 · SVR for loss severity on defaulted loans

1. Load `loan_features.parquet`; keep defaulted loans (`default_12m == 1`); target `log_loss = log1p(loss_amount_aed)`.
2. Features: log loan amount, product, tenure, interest rate, log salary, DBR, bureau score. Build a scaled pipeline.
3. Fit a `LinearRegression` baseline and tune `SVR(kernel='rbf')` over C ∈ {1, 10}, ε ∈ {0.05, 0.2}, γ ∈ {'scale', 0.05} on a 6k train sample (3-fold CV).
4. Report MAE in AED and MAPE on the out-of-time defaults (`split == 'test'`).

In [ ]:
# Your code for step 5


## Step 6 · Record the bake-off

1. Write `svm_results.json` with the fraud board, the timings and the SVR results.

In [ ]:
# Your code for step 6
